# Netflix Audience Rating Classifier
## Auspify Technologies – Machine Learning Internship | Task 3

**Objective:** Build, evaluate, and tune multi-class classification models (Logistic Regression, Decision Tree, Random Forest) to predict audience rating categories (`TV-MA`, `TV-14`, `R`, `PG-13`, `TV-PG`, etc.) on Netflix content using content attributes.

**Dataset Basis:** Exclusively uses Auspify Technologies-provided `Dataset.csv` containing 8,790 records and 10 attributes.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization styling
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Add parent directory for modular imports
sys.path.append(os.path.abspath('..'))
from ml.preprocess import NetflixPreprocessor
from ml.tune import tune_decision_tree, tune_random_forest
from ml.evaluate import evaluate_model, extract_feature_importances

## 1. Data Understanding & Exploratory Data Analysis (EDA)
Let us load the provided dataset and inspect the columns, data types, missing values, and target distribution.

In [2]:
df = pd.read_csv('../data/Dataset.csv')
print(f'Dataset Shape: {df.shape}')
print(df.info())

In [3]:
# Target Variable Distribution
plt.figure(figsize=(12, 5))
order = df['rating'].value_counts().index
sns.countplot(data=df, x='rating', order=order, palette='viridis')
plt.title('Distribution of Audience Ratings on Netflix (Auspify Dataset)', fontsize=14, fontweight='bold')
plt.xlabel('Rating Category')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [4]:
# Content Type breakdown by Rating
plt.figure(figsize=(14, 6))
sns.countplot(data=df, x='rating', hue='type', order=order, palette=['#E50914', '#221F1F'])
plt.title('Audience Ratings by Content Type (Movies vs TV Shows)', fontsize=14, fontweight='bold')
plt.xlabel('Rating Category')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.legend(title='Type')
plt.tight_layout()
plt.show()

## 2. Feature Engineering & Preprocessing
- Exclude `show_id` (identifier) and `title` (avoid memorization).
- Parse `date_added` into `added_year`, `added_month`, `added_day`.
- Extract `duration_value` and `duration_type` from `duration`.
- Extract `country_count` and primary country representation.
- Derive director presence and frequent director encoding.
- Generate multi-hot indicators for all 42 unique Netflix genres.
- Standardize numerical features and one-hot encode categorical features.

In [5]:
from sklearn.model_selection import train_test_split

X_raw = df.drop(columns=['rating', 'show_id', 'title'])
y_raw = df['rating']

# Stratified Train/Test Split
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

preprocessor = NetflixPreprocessor()
X_train = preprocessor.fit_transform(X_train_raw, y_train_raw)
X_test = preprocessor.transform(X_test_raw)

y_train = preprocessor.encode_target(y_train_raw)
y_test = preprocessor.encode_target(y_test_raw)
class_names = preprocessor.get_classes()

print(f'Training Matrix: {X_train.shape}')
print(f'Testing Matrix: {X_test.shape}')
print(f'Classes: {class_names}')

## 3. Baseline Model Training
Training Baseline Logistic Regression, Decision Tree Classifier, and Random Forest Classifier.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# 1. Logistic Regression Baseline
lr = LogisticRegression(max_iter=2000, random_state=42)
lr.fit(X_train, y_train)
eval_lr = evaluate_model(lr, X_test, y_test, class_names)
print(f"Logistic Regression Baseline -> Accuracy: {eval_lr['accuracy']:.4f}, Weighted F1: {eval_lr['f1_weighted']:.4f}")

# 2. Decision Tree Baseline
dt_base = DecisionTreeClassifier(random_state=42)
dt_base.fit(X_train, y_train)
eval_dt = evaluate_model(dt_base, X_test, y_test, class_names)
print(f"Decision Tree Baseline -> Accuracy: {eval_dt['accuracy']:.4f}, Weighted F1: {eval_dt['f1_weighted']:.4f}")

# 3. Random Forest Baseline
rf_base = RandomForestClassifier(n_estimators=100, random_state=42)
rf_base.fit(X_train, y_train)
eval_rf = evaluate_model(rf_base, X_test, y_test, class_names)
print(f"Random Forest Baseline -> Accuracy: {eval_rf['accuracy']:.4f}, Weighted F1: {eval_rf['f1_weighted']:.4f}")

## 4. Hyperparameter Tuning with 5-Fold Cross-Validation
Optimizing Decision Tree with GridSearchCV and Random Forest with RandomizedSearchCV.

In [7]:
# Tune Decision Tree
dt_tuned, dt_params, _ = tune_decision_tree(X_train, y_train, cv=5)
eval_dt_tuned = evaluate_model(dt_tuned, X_test, y_test, class_names)
print(f"Decision Tree Tuned -> Accuracy: {eval_dt_tuned['accuracy']:.4f}, Weighted F1: {eval_dt_tuned['f1_weighted']:.4f}")

# Tune Random Forest
rf_tuned, rf_params, _ = tune_random_forest(X_train, y_train, cv=5, n_iter=15)
eval_rf_tuned = evaluate_model(rf_tuned, X_test, y_test, class_names)
print(f"Random Forest Tuned -> Accuracy: {eval_rf_tuned['accuracy']:.4f}, Weighted F1: {eval_rf_tuned['f1_weighted']:.4f}")

## 5. Comparative Evaluation & Confusion Matrix
Comparing all models side-by-side and displaying the confusion matrix for the best model.

In [8]:
comparison_df = pd.DataFrame([
    {'Model': 'Logistic Regression (Baseline)', 'Accuracy': eval_lr['accuracy'], 'Weighted F1': eval_lr['f1_weighted'], 'Weighted Precision': eval_lr['precision_weighted'], 'Weighted Recall': eval_lr['recall_weighted']},
    {'Model': 'Decision Tree (Baseline)', 'Accuracy': eval_dt['accuracy'], 'Weighted F1': eval_dt['f1_weighted'], 'Weighted Precision': eval_dt['precision_weighted'], 'Weighted Recall': eval_dt['recall_weighted']},
    {'Model': 'Decision Tree (Tuned)', 'Accuracy': eval_dt_tuned['accuracy'], 'Weighted F1': eval_dt_tuned['f1_weighted'], 'Weighted Precision': eval_dt_tuned['precision_weighted'], 'Weighted Recall': eval_dt_tuned['recall_weighted']},
    {'Model': 'Random Forest (Baseline)', 'Accuracy': eval_rf['accuracy'], 'Weighted F1': eval_rf['f1_weighted'], 'Weighted Precision': eval_rf['precision_weighted'], 'Weighted Recall': eval_rf['recall_weighted']},
    {'Model': 'Random Forest (Tuned)', 'Accuracy': eval_rf_tuned['accuracy'], 'Weighted F1': eval_rf_tuned['f1_weighted'], 'Weighted Precision': eval_rf_tuned['precision_weighted'], 'Weighted Recall': eval_rf_tuned['recall_weighted']}
])
print(comparison_df.to_string(index=False))

In [9]:
# Confusion Matrix Heatmap for Best Model
plt.figure(figsize=(10, 8))
cm = np.array(eval_rf_tuned['confusion_matrix'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Tuned Random Forest (Test Set)', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Rating')
plt.ylabel('Actual Rating')
plt.tight_layout()
plt.show()

In [10]:
# Feature Importances Plot
feat_names = preprocessor.get_feature_names()
importances = rf_tuned.feature_importances_
top_indices = np.argsort(importances)[::-1][:15]

plt.figure(figsize=(12, 6))
sns.barplot(x=importances[top_indices], y=[feat_names[i] for i in top_indices], palette='mako')
plt.title('Top 15 Feature Importances (Tuned Random Forest)', fontsize=14, fontweight='bold')
plt.xlabel('Relative Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()